In [160]:
import re
import pandas as pd
from scipy import stats
import re
import pandas as pd
from pathlib import Path

In [161]:
def normalize_text(text):
    """Normalize text: NFKD, remove accents, lowercase, remove parentheses content"""
    return text.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.lower().apply(lambda x: re.sub(r'\s*\([^)]*\)', '', x))


def load_and_prepare_data(interdis_path, institutions_path, label_col_name):
    """
    Load interdis CSV, rename Unnamed: 0 to ranking_position, 
    merge with institutions names, and normalize institution names.
    
    Args:
        interdis_path: path to sorted_interdis.csv
        institutions_path: path to institutions_names.csv
        label_col_name: column name to use for normalized names (e.g., 'LABEL' or 'normalized_Universidade')
    
    Returns:
        DataFrame with ranking_position, LABEL, and normalized_name columns
    """
    # Load interdis data and rename index column
    df = pd.read_csv(interdis_path, encoding="utf-8")
    df.rename(columns={'Unnamed: 0': 'ranking_position'}, inplace=True)
    
    # Load and merge institutions names
    institutions = pd.read_csv(institutions_path, encoding="utf-8")
    df['LABEL'] = institutions['ins_name']
    
    # Normalize institution names
    df['normalized_name'] = normalize_text(df['LABEL'])
    
    return df

def load_all_subfolders(parent_folder):
    """
    Load and prepare data from all subfolders containing sorted_interdis.csv and institutions_names.csv
    
    Args:
        parent_folder: path to parent folder (e.g., './data/macro' or './data/sub')
    
    Returns:
        Dictionary with subfolder names as keys and prepared DataFrames as values
    """
    data = {}
    parent_path = Path(parent_folder)
    
    # Find all subdirectories with sorted_interdis.csv
    for subfolder in sorted(parent_path.glob('*')):
        if subfolder.is_dir():
            interdis_file = subfolder / 'sorted_interdis.csv'
            institutions_file = subfolder / 'institutions_names.csv'
            
            if interdis_file.exists() and institutions_file.exists():
                try:
                    df = load_and_prepare_data(
                        str(interdis_file),
                        str(institutions_file),
                        'LABEL'
                    )
                    data[subfolder.name] = df
                    print(f"✓ Loaded {subfolder.name}: {len(df)} rows")
                except Exception as e:
                    print(f"✗ Error loading {subfolder.name}: {e}")
    
    return data

In [162]:
print("Loading macro subfolders:")
macro_data = load_all_subfolders('./data/macro')

print("\nLoading sub subfolders:")
sub_data = load_all_subfolders('./data/sub')

ruf = pd.read_csv('./data/RUF.csv', encoding="utf-8", delimiter=';')
ruf['normalized_Universidade'] = normalize_text(ruf['Universidade'])

print(f"\n✓ Loaded {len(macro_data)} macro subfolders and {len(sub_data)} sub subfolders")

Loading macro subfolders:
✓ Loaded macro_no_quality: 249 rows
✓ Loaded macro_q1: 249 rows
✓ Loaded macro_q2: 249 rows
✓ Loaded macro_q3: 249 rows
✓ Loaded macro_q4: 249 rows
✓ Loaded macro_quartile: 249 rows

Loading sub subfolders:
✓ Loaded sub_no_quality: 249 rows
✓ Loaded sub_q1: 249 rows
✓ Loaded sub_q2: 249 rows
✓ Loaded sub_q3: 249 rows
✓ Loaded sub_q4: 249 rows
✓ Loaded sub_quartile: 249 rows

✓ Loaded 6 macro subfolders and 6 sub subfolders


In [163]:
# Merge each macro subfolder with RUF data
merged_macro_data = {}
for name, df in macro_data.items():
    merged = df.merge(ruf, left_on='normalized_name', right_on='normalized_Universidade', how='left')
    merged_macro_data[name] = merged
    print(f"✓ Merged {name}: {len(merged)} rows")

# Merge each sub subfolder with RUF data
merged_sub_data = {}
for name, df in sub_data.items():
    merged = df.merge(ruf, left_on='normalized_name', right_on='normalized_Universidade', how='left')
    merged_sub_data[name] = merged
    print(f"✓ Merged {name}: {len(merged)} rows")

print(f"\n✓ Total merged: {len(merged_macro_data)} macro subfolders and {len(merged_sub_data)} sub subfolders")


✓ Merged macro_no_quality: 249 rows
✓ Merged macro_q1: 249 rows
✓ Merged macro_q2: 249 rows
✓ Merged macro_q3: 249 rows
✓ Merged macro_q4: 249 rows
✓ Merged macro_quartile: 249 rows
✓ Merged sub_no_quality: 249 rows
✓ Merged sub_q1: 249 rows
✓ Merged sub_q2: 249 rows
✓ Merged sub_q3: 249 rows
✓ Merged sub_q4: 249 rows
✓ Merged sub_quartile: 249 rows

✓ Total merged: 6 macro subfolders and 6 sub subfolders


In [164]:
# Compute Kendall Tau for each macro subfolder
print("=== MACRO KENDALL TAU RESULTS ===\n")
macro_kendall_results = {}

for name, merged in merged_macro_data.items():
    merged['Ranking_num'] = pd.to_numeric(merged['Ranking'], errors='coerce')
    merged['DIV_STAR'] = pd.to_numeric(merged['ranking_position'], errors='coerce')
    
    mask = merged['Ranking_num'].notna() & merged['ranking_position'].notna()
    x = merged.loc[mask, 'Ranking_num']
    y = merged.loc[mask, 'ranking_position']
    
    print(f"{name}:")
    print(f"  Paired observations: {len(x)}")
    
    if len(x) >= 2:
        tau, p_value = stats.kendalltau(x, y, nan_policy='omit')
        macro_kendall_results[name] = {'tau': tau, 'p_value': p_value, 'n': len(x)}
        print(f"  Kendall Tau: {tau:.6f}")
        print(f"  p-value: {p_value}")
    else:
        macro_kendall_results[name] = {'tau': None, 'p_value': None, 'n': len(x)}
        print(f"  Not enough paired observations (need >=2)")
    print()


=== MACRO KENDALL TAU RESULTS ===

macro_no_quality:
  Paired observations: 166
  Kendall Tau: 0.448361
  p-value: 1.0934412857457732e-17

macro_q1:
  Paired observations: 164
  Kendall Tau: 0.608631
  p-value: 7.1154483842117225e-31

macro_q2:
  Paired observations: 168
  Kendall Tau: 0.642921
  p-value: 4.554554436061084e-35

macro_q3:
  Paired observations: 169
  Kendall Tau: 0.665162
  p-value: 1.2451780132121307e-37

macro_q4:
  Paired observations: 170
  Kendall Tau: 0.657164
  p-value: 5.478140620155197e-37

macro_quartile:
  Paired observations: 166
  Kendall Tau: 0.448361
  p-value: 1.0934412857457732e-17



In [165]:
# Compute Kendall Tau for each sub subfolder
print("=== SUB KENDALL TAU RESULTS ===\n")
sub_kendall_results = {}

for name, merged in merged_sub_data.items():
    merged['Ranking_num'] = pd.to_numeric(merged['Ranking'], errors='coerce')
    merged['ranking_position'] = pd.to_numeric(merged['ranking_position'], errors='coerce')
    
    mask = merged['Ranking_num'].notna() & merged['ranking_position'].notna()
    x = merged.loc[mask, 'Ranking_num']
    y = merged.loc[mask, 'ranking_position']
    
    print(f"{name}:")
    print(f"  Paired observations: {len(x)}")
    
    if len(x) >= 2:
        tau, p_value = stats.kendalltau(x, y, nan_policy='omit')
        sub_kendall_results[name] = {'tau': tau, 'p_value': p_value, 'n': len(x)}
        print(f"  Kendall Tau: {tau:.6f}")
        print(f"  p-value: {p_value}")
    else:
        sub_kendall_results[name] = {'tau': None, 'p_value': None, 'n': len(x)}
        print(f"  Not enough paired observations (need >=2)")
    print()


=== SUB KENDALL TAU RESULTS ===

sub_no_quality:
  Paired observations: 167
  Kendall Tau: 0.711144
  p-value: 2.855843114493094e-42

sub_q1:
  Paired observations: 165
  Kendall Tau: 0.735199
  p-value: 1.574351313411101e-44

sub_q2:
  Paired observations: 168
  Kendall Tau: 0.723438
  p-value: 6.095721449156107e-44

sub_q3:
  Paired observations: 168
  Kendall Tau: 0.735430
  p-value: 2.37116940338811e-45

sub_q4:
  Paired observations: 171
  Kendall Tau: 0.718631
  p-value: 3.8151213582329823e-44

sub_quartile:
  Paired observations: 167
  Kendall Tau: 0.711144
  p-value: 2.855843114493094e-42



In [ ]:
# Build a datatable of paired merged positions and RUF positions,
# plus a summary table with Kendall Tau and p-values per folder.
from IPython.display import display

paired_frames = []
summary_rows = []

def process_merged_dict(merged_dict, group_type):
    for name, merged in merged_dict.items():
        df = merged.copy()
        # Ensure numeric columns exist
        df['Ranking_num'] = pd.to_numeric(df.get('Ranking', df.get('Ranking_num')), errors='coerce')
        df['ranking_position_num'] = pd.to_numeric(df.get('ranking_position'), errors='coerce')

        mask = df['Ranking_num'].notna() & df['ranking_position_num'].notna()
        paired = df.loc[mask, ['LABEL', 'normalized_name', 'Ranking_num', 'ranking_position_num']].copy()
        paired.rename(columns={'Ranking_num': 'ruf_ranking', 'ranking_position_num': 'merged_ranking'}, inplace=True)
        paired['source'] = f'{group_type}/{name}'

        # Compute group-level Kendall Tau if we have enough pairs
        if len(paired) >= 2:
            tau, p_value = stats.kendalltau(paired['ruf_ranking'], paired['merged_ranking'], nan_policy='omit')
        else:
            tau, p_value = (None, None)

        paired['group_tau'] = tau
        paired['group_p_value'] = p_value
        paired['group_n'] = len(paired)

        paired_frames.append(paired)
        summary_rows.append({'source': f'{group_type}/{name}', 'tau': tau, 'p_value': p_value, 'n': len(paired)})

# Process macro and sub merged dictionaries created earlier in the notebook
process_merged_dict(merged_macro_data, 'macro')
process_merged_dict(merged_sub_data, 'sub')

# Concatenate results
if paired_frames:
    paired_df = pd.concat(paired_frames, ignore_index=True)
else:
    paired_df = pd.DataFrame(columns=['LABEL', 'normalized_name', 'ruf_ranking', 'merged_ranking', 'source', 'group_tau', 'group_p_value', 'group_n'])

kendall_summary = pd.DataFrame(summary_rows)

# Reorder/rename columns for clarity
paired_df = paired_df[['source', 'LABEL', 'normalized_name', 'merged_ranking', 'ruf_ranking', 'group_tau', 'group_p_value', 'group_n']]
paired_df.rename(columns={'merged_ranking': 'merged_position', 'ruf_ranking': 'ruf_position'}, inplace=True)

# Display concise summary and first rows of paired table
print('=== Kendall summary (per folder) ===')
display(kendall_summary.sort_values('source').reset_index(drop=True))

print(f'✓ Saved kendall summary: {len(kendall_summary)} rows to ./data/kendall_summary.csv')
print(f'✓ Saved paired rows: {len(paired_df)} rows to ./data/paired_rankings.csv')
kendall_summary.to_csv('./data/kendall_summary.csv', index=False, encoding='utf-8')
paired_df.to_csv('./data/paired_rankings.csv', index=False, encoding='utf-8')

SyntaxError: invalid syntax (2983896514.py, line 53)